# Logistic model: stationary figures

This notebook reproduces the stationary logistic-model figures in the intensity variable $I$ from compact plot data by default. Set `USE_PRECOMPUTED_PLOT_DATA=False` and enable the simulation flags to regenerate raw data. The stationary corrections are generated symbolically from the model drift with `ucna_utils.py`.


In [ ]:
from dataclasses import asdict, dataclass
from functools import lru_cache
from pathlib import Path
import pickle

import matplotlib as mpl
import matplotlib.pyplot as plt
import numba as nb
import numpy as np
import pandas as pd
import seaborn as sns
import sympy as sp
from scipy.integrate import cumulative_trapezoid, quad
from tqdm import tqdm

plt.rcParams.update({
    'axes.labelsize': 11,
    'axes.labelweight': 'bold',
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,    
})

from ucna_utils import (
    condition_on_positive_theory, density_interval, generate_stationary_correction,
    kl_divergence,
    replica_raw_kl_uncertainty,
    ucna_stationary_unnormalized,
)

RUN_SIMULATIONS = False
OUTPUT_DIR = Path('results/logistic_stationary')
USE_PRECOMPUTED_PLOT_DATA = True
PLOT_DATA_DIR = Path('plot_data')


In [ ]:
@nb.njit
def logistic_drift(x):
    return 2.0 - 2.0 * np.exp(x)

def logistic_drift_prime(x):
    return -2.0 * np.exp(x)

D_PRODUCTION = 2.0
TAUS = [0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
TAIL_TOLERANCE = 1e-8

def logistic_ucna_density_x(x, tau, D):
    # Closed form of the same UCNA density used below. The omitted
    # integration constant cancels upon normalization.
    ex = np.exp(x)
    force = 2.0 * (1.0 - ex)
    gamma = 1.0 + 2.0 * tau * ex
    exponent = (-0.5 * tau * force**2 + 2.0*x - 2.0*ex) / D
    return gamma * np.exp(exponent)

def logistic_intensity_interval(tau, D, tail_tolerance=TAIL_TOLERANCE):
    lower_x, upper_x = density_interval(
        lambda x: logistic_ucna_density_x(x, tau, D),
        tail_tolerance=tail_tolerance,
    )
    return float(np.exp(lower_x)), float(np.exp(upper_x))


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    print('Symbolic corrections are generated only in raw-analysis mode.')
else:
    CORRECTION_VARIABLE = sp.symbols('x', real=True)

    @lru_cache(maxsize=None)
    def generated_logistic_correction(tau, D):
        drift = 2 * (1 - sp.exp(CORRECTION_VARIABLE))
        return generate_stationary_correction(
            drift, CORRECTION_VARIABLE, float(tau), float(D)
        )

    LOGISTIC_CORRECTION_KEYS = [
        (float(tau), float(D_PRODUCTION)) for tau in TAUS
    ]
    CORRECTIONS = {
        key: generated_logistic_correction(*key)
        for key in tqdm(LOGISTIC_CORRECTION_KEYS, desc='Generating symbolic corrections')
    }
    print(f'Generated {len(CORRECTIONS)} logistic corrections')


## Production configurations and simulation


In [ ]:
@dataclass(frozen=True)
class LogisticExperiment:
    tau: float
    D: float = D_PRODUCTION
    dt: float = 0.01
    duration: float = 10
    n_replicas: int = 10
    samples_per_replica: int = 1000000
    n_bins: int = 200
    I_min: float | None = None
    I_max: float | None = None
    base_seed: int = 20260812
    initial_I: float = 0.1

    def __post_init__(self):
        if (self.I_min is None) != (self.I_max is None):
            raise ValueError('I_min and I_max must both be given or both omitted')
        if self.I_min is None:
            lower, upper = logistic_intensity_interval(self.tau, self.D)
            object.__setattr__(self, 'I_min', lower)
            object.__setattr__(self, 'I_max', upper)

    @property
    def n_steps(self):
        steps = int(round(self.duration / self.dt))
        if not np.isclose(steps * self.dt, self.duration):
            raise ValueError('duration must be an integer multiple of dt')
        return steps

    @property
    def total_samples(self):
        return self.n_replicas * self.samples_per_replica

PRODUCTION_CONFIGS = [LogisticExperiment(tau=tau) for tau in TAUS]
config_table = pd.DataFrame(asdict(config) for config in PRODUCTION_CONFIGS)
config_table['trajectory_steps_total'] = (
    config_table.n_replicas * config_table.samples_per_replica
    * (config_table.duration / config_table.dt).astype(int)
)
display(config_table)
print(f"{config_table.trajectory_steps_total.sum():,.0f} total trajectory steps")


In [ ]:
def replica_seed(config, replica):
    sequence = np.random.SeedSequence([
        config.base_seed, int(round(config.tau * 1_000_000)),
        int(round(config.D * 1_000_000)), replica,
    ])
    return int(sequence.generate_state(1, dtype=np.uint32)[0])

def intensity_edges(config):
    return np.linspace(config.I_min, config.I_max, config.n_bins + 1)

@nb.njit
def simulate_logistic_endpoints(D, tau, dt, n_steps, n_realizations, seed, initial_I=0.1):
    np.random.seed(seed)
    rho = np.exp(-dt / tau)
    sigma = np.sqrt(D / tau)
    noise_scale = np.sqrt(1.0 - rho**2) * sigma
    intensity = np.full(n_realizations, initial_I, dtype=np.float64)
    eta = np.empty(n_realizations, dtype=np.float64)
    for realization in range(n_realizations):
        eta[realization] = sigma * np.random.normal()
    for _ in range(n_steps):
        for realization in range(n_realizations):
            value = intensity[realization]
            force = 2.0 * value * (1.0 - value) + value * eta[realization]
            predictor = value + dt * force
            eta[realization] = rho * eta[realization] + noise_scale * np.random.normal()
            corrected_force = (
                2.0 * predictor * (1.0 - predictor)
                + predictor * eta[realization]
            )
            intensity[realization] = value + 0.5 * dt * (force + corrected_force)
    return intensity

def histogram_endpoints(I_by_replica, config):
    edges = intensity_edges(config)
    counts = np.asarray([
        np.histogram(I_values, bins=edges)[0]
        for I_values in I_by_replica
    ])
    outside = np.asarray([
        len(I_values) - row.sum() for I_values, row in zip(I_by_replica, counts)
    ])
    return edges, counts, outside

def run_config(config):
    endpoints, seeds = [], []
    for replica in tqdm(range(config.n_replicas),
                        desc=f'replicas: tau={config.tau:g}', leave=False):
        seed = replica_seed(config, replica)
        final_I = simulate_logistic_endpoints(
            config.D, config.tau, config.dt, config.n_steps,
            config.samples_per_replica, seed, config.initial_I,
        )
        endpoints.append(final_I)
        seeds.append(seed)
    endpoints = np.asarray(endpoints)
    edges, counts, outside = histogram_endpoints(endpoints, config)
    return {
        'schema_version': 2, 'config': asdict(config), 'edges': edges,
        'replica_endpoints_I': endpoints, 'replica_counts': counts,
        'replica_seeds': np.asarray(seeds),
        'replica_seed_chunks': [[int(seed)] for seed in seeds],
        'replica_chunk_sizes': [[config.samples_per_replica] for _ in seeds],
        'outside_counts': outside,
    }

def result_filename(config):
    return OUTPUT_DIR / f'stationary_tau_{config.tau:g}_D_{config.D:g}.pkl'

def rebin_result(result, config):
    edges, counts, outside = histogram_endpoints(
        np.asarray(result['replica_endpoints_I']), config
    )
    updated = dict(result)
    updated.update(config=asdict(config), edges=edges,
                   replica_counts=counts, outside_counts=outside)
    return updated

def supplemental_seed(config, replica, previous_sample_count):
    sequence = np.random.SeedSequence([
        config.base_seed, int(round(config.tau * 1_000_000)),
        int(round(config.D * 1_000_000)), replica,
        0xA5A5A5A5, int(previous_sample_count),
    ])
    return int(sequence.generate_state(1, dtype=np.uint32)[0])

def extend_result(saved, target_config):
    old_config = dict(saved['config'])
    target = asdict(target_config)
    fixed_fields = set(target) - {
        'n_replicas', 'samples_per_replica', 'n_bins', 'I_min', 'I_max'
    }
    changed_fixed = {
        key: (old_config.get(key), target[key]) for key in fixed_fields
        if old_config.get(key) != target[key]
    }
    if changed_fixed:
        raise ValueError(f'Non-sample configuration changes cannot be appended: {changed_fixed}')

    old_replicas = int(old_config['n_replicas'])
    old_samples = int(old_config['samples_per_replica'])
    if target_config.n_replicas < old_replicas:
        raise ValueError('n_replicas cannot be reduced in an existing result')
    if target_config.samples_per_replica < old_samples:
        raise ValueError('samples_per_replica cannot be reduced in an existing result')

    endpoints = [np.asarray(row) for row in saved['replica_endpoints_I']]
    if len(endpoints) != old_replicas or any(len(row) != old_samples for row in endpoints):
        raise ValueError('Saved endpoint dimensions disagree with the saved configuration')
    primary_seeds = [int(seed) for seed in saved['replica_seeds']]
    if 'replica_seed_chunks' in saved:
        seed_chunks = [list(map(int, row)) for row in saved['replica_seed_chunks']]
        chunk_sizes = [list(map(int, row)) for row in saved['replica_chunk_sizes']]
    else:
        seed_chunks = [[seed] for seed in primary_seeds]
        chunk_sizes = [[old_samples] for _ in range(old_replicas)]

    work = []
    if target_config.samples_per_replica > old_samples:
        work.extend((replica, target_config.samples_per_replica-old_samples)
                    for replica in range(old_replicas))
    work.extend((replica, target_config.samples_per_replica)
                for replica in range(old_replicas, target_config.n_replicas))

    for replica, n_new in tqdm(
        work, desc=f'extending tau={target_config.tau:g}', leave=False,
    ):
        if replica < old_replicas:
            seed = supplemental_seed(target_config, replica, len(endpoints[replica]))
        else:
            seed = replica_seed(target_config, replica)
        new_endpoints = simulate_logistic_endpoints(
            target_config.D, target_config.tau, target_config.dt,
            target_config.n_steps, n_new, seed, target_config.initial_I,
        )
        if replica < old_replicas:
            endpoints[replica] = np.concatenate([endpoints[replica], new_endpoints])
            seed_chunks[replica].append(seed)
            chunk_sizes[replica].append(n_new)
        else:
            endpoints.append(new_endpoints)
            primary_seeds.append(seed)
            seed_chunks.append([seed])
            chunk_sizes.append([n_new])

    endpoints = np.asarray(endpoints)
    edges, counts, outside = histogram_endpoints(endpoints, target_config)
    return {
        'schema_version': 2, 'config': target, 'edges': edges,
        'replica_endpoints_I': endpoints, 'replica_counts': counts,
        'replica_seeds': np.asarray(primary_seeds),
        'replica_seed_chunks': seed_chunks, 'replica_chunk_sizes': chunk_sizes,
        'outside_counts': outside,
    }

def run_missing_configs(configs):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    actions = []
    reusable = 0
    histogram_fields = {'n_bins', 'I_min', 'I_max'}
    sample_fields = {'n_replicas', 'samples_per_replica'}
    for config in tqdm(configs, desc='checking saved configurations'):
        filename = result_filename(config)
        if not filename.exists():
            actions.append(('new', config, filename, None))
            continue
        with filename.open('rb') as handle:
            saved = pickle.load(handle)
        if 'replica_endpoints_I' not in saved:
            actions.append(('stale', config, filename, None))
            continue
        old, target = saved['config'], asdict(config)
        changed = {key for key in target if old.get(key) != target[key]}
        unsupported = changed - histogram_fields - sample_fields
        decreases = (
            target['n_replicas'] < old['n_replicas']
            or target['samples_per_replica'] < old['samples_per_replica']
        )
        if unsupported or decreases:
            raise ValueError(
                f'{filename} differs in non-appendable settings: '
                f'fields={sorted(unsupported)}, decrease={decreases}'
            )
        if changed & sample_fields:
            actions.append(('extend', config, filename, saved))
        elif changed & histogram_fields:
            actions.append(('rebin', config, filename, saved))
        else:
            reusable += 1

    counts = {kind: sum(item[0] == kind for item in actions)
              for kind in ['new', 'stale', 'rebin', 'extend']}
    print(f"{reusable} reusable, {counts['extend']} extendable, "
          f"{counts['rebin']} rebin-only, {counts['new']} new, "
          f"{counts['stale']} stale")
    for action, config, filename, saved in tqdm(actions, desc='logistic configurations'):
        print(f'{action}: tau={config.tau:g}, D={config.D:g}')
        if action == 'extend':
            result = extend_result(saved, config)
        elif action == 'rebin':
            result = rebin_result(saved, config)
        else:
            result = run_config(config)
        temporary = filename.with_suffix('.pkl.tmp')
        with temporary.open('wb') as handle:
            pickle.dump(result, handle)
        temporary.replace(filename)

def load_results():
    loaded = {}
    for filename in sorted(OUTPUT_DIR.glob('stationary_tau_*_D_*.pkl')):
        with filename.open('rb') as handle:
            result = pickle.load(handle)
        loaded[(result['config']['tau'], result['config']['D'])] = result
    return loaded


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    print('Using packaged logistic plot data; raw results are not loaded.')
else:
    if RUN_SIMULATIONS:
        run_missing_configs(PRODUCTION_CONFIGS)

    stationary_data = load_results()
    print(f'Loaded {len(stationary_data)}/{len(PRODUCTION_CONFIGS)} configurations')


## Theoretical bin probabilities and KL analysis


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    print('Theory is evaluated only in raw-analysis mode.')
else:
    def integrate_log_density_on_intensity_bins(density_x, I_edges, points=65):
        weights = []
        for lower_I, upper_I in zip(I_edges[:-1], I_edges[1:]):
            x = np.linspace(np.log(lower_I), np.log(upper_I), points)
            values = np.asarray([density_x(value) for value in x], dtype=float)
            weights.append(getattr(np, 'trapezoid', np.trapz)(values, x))
        weights = np.asarray(weights)
        if np.any(~np.isfinite(weights)) or weights.sum() <= 0:
            raise ValueError('theory produced invalid bin weights')
        return weights / weights.sum()

    def ucna_bin_probabilities(tau, D, I_edges):
        lower_x = np.log(I_edges[0])
        return integrate_log_density_on_intensity_bins(
            lambda x: ucna_stationary_unnormalized(
                x, tau, D, logistic_drift.py_func, logistic_drift_prime,
                lower_bound=lower_x,
            ), I_edges,
        )

    def Dbfpe(x, tau, D):
        """Logistic cBFPE diffusion from the backward-flow integral."""
        values = np.asarray(x, dtype=float)
        intensity = np.exp(values)
        diffusion = np.empty_like(intensity)
        below = intensity <= 1.0
        diffusion[below] = D * (
            1.0 - intensity[below] + intensity[below] / (1.0 + 2.0*tau)
        )
        above = ~below
        if np.any(above):
            current_I = intensity[above]
            u_max = np.log(current_I / (current_I - 1.0)) / (2.0*tau)
            diffusion[above] = D * (
                (1.0-current_I) * (-np.expm1(-u_max))
                + current_I/(1.0+2.0*tau)
                  * (-np.expm1(-(1.0+2.0*tau)*u_max))
            )
        if np.any(~np.isfinite(diffusion)) or np.any(diffusion <= 0):
            raise ValueError('logistic cBFPE diffusion is nonpositive or nonfinite')
        return float(diffusion) if values.ndim == 0 else diffusion

    def bfpe_stationary(x, tau, D, lower_bound):
        """Unnormalized zero-current cBFPE stationary density in x=log(I)."""
        exponent = quad(
            lambda y: logistic_drift.py_func(y) / Dbfpe(y, tau, D),
            lower_bound, x, points=[0.0] if lower_bound < 0.0 < x else None,
            limit=200,
        )[0]
        return np.exp(exponent) / Dbfpe(x, tau, D)

    def bfpe_bin_probabilities(tau, D, I_edges, points=65):
        """Integrate the cBFPE stationary density on fixed intensity bins."""
        x_edges = np.log(np.asarray(I_edges, dtype=float))
        pieces = [np.linspace(a, b, points) for a, b in zip(x_edges[:-1], x_edges[1:])]
        x_grid = np.concatenate([pieces[0]] + [piece[1:] for piece in pieces[1:]])
        diffusion = Dbfpe(x_grid, tau, D)
        exponent = cumulative_trapezoid(
            logistic_drift.py_func(x_grid) / diffusion, x_grid, initial=0.0
        )
        log_density = exponent - np.log(diffusion)
        density = np.exp(log_density - np.max(log_density))
        weights = np.empty(len(I_edges)-1)
        for index in range(len(weights)):
            start = index * (points-1)
            stop = start + points
            weights[index] = getattr(np, 'trapezoid', np.trapz)(
                density[start:stop], x_grid[start:stop]
            )
        if np.any(~np.isfinite(weights)) or np.any(weights < 0) or weights.sum() <= 0:
            raise ValueError('logistic cBFPE produced invalid bin weights')
        return weights / weights.sum()

    def corrected_bin_probabilities(tau, D, I_edges):
        correction = CORRECTIONS.get((float(tau), float(D)))
        if correction is None:
            return None, np.nan
        signed_weights = integrate_log_density_on_intensity_bins(correction, I_edges)
        # Diagnose the perturbative correction before enforcing positivity.
        negative_mass = max(0.0, float(-signed_weights[signed_weights < 0].sum()))
        # KL requires a probability distribution: project small negative bin weights
        # onto the probability simplex by clipping, then renormalize.
        weights = np.maximum(signed_weights, 0.0)
        positive_mass = weights.sum()
        if not np.isfinite(positive_mass) or positive_mass <= 0:
            raise ValueError('correction has no positive probability mass')
        return weights / positive_mass, negative_mass


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    results = pd.read_csv(PLOT_DATA_DIR / 'logistic_kl.csv')
else:
    NEGATIVE_MASS_THRESHOLD = 1e-3
    EXCLUDED_EMPIRICAL_MASS_THRESHOLD = 1e-3

    def summarize_approximation(result, theory, counts_by_replica=None, n_total=None):
        if counts_by_replica is None:
            counts_by_replica = np.asarray(result['replica_counts'])
        pooled = counts_by_replica.sum(axis=0)
        if n_total is None:
            config = result['config']
            n_total = config['n_replicas'] * config['samples_per_replica']
        raw_mean, raw_sem, uncertainty = replica_raw_kl_uncertainty(
            counts_by_replica, theory, n_total
        )
        return {
            'KL': n_total / (len(pooled) - 1) * kl_divergence(
                pooled, theory
            ),
            'KL_uncertainty': uncertainty,
            'replica_raw_KL_mean': raw_mean,
            'replica_raw_KL_sem': raw_sem,
        }

    rows = []
    for (tau, D), result in tqdm(stationary_data.items(), desc='stationary KL values'):
        outside = int(np.asarray(result['outside_counts']).sum())
        ucna = ucna_bin_probabilities(tau, D, result['edges'])
        row = summarize_approximation(result, ucna)
        row.update(tau=tau, D=D, Approximation='UCNA', outside=outside,
                   tau2D=D*tau**2, negative_mass=0.0, correction_valid=True)
        rows.append(row)
        bfpe = bfpe_bin_probabilities(tau, D, result['edges'])
        row = summarize_approximation(result, bfpe)
        row.update(tau=tau, D=D, Approximation='cBFPE', outside=outside,
                   tau2D=D*tau**2, negative_mass=0.0, correction_valid=True)
        rows.append(row)
        corrected, negative_mass = corrected_bin_probabilities(tau, D, result['edges'])
        if corrected is not None:
            conditioned_counts, conditioned_correction, excluded_fraction = (
                condition_on_positive_theory(result['replica_counts'], corrected)
            )
            retained_samples = int(conditioned_counts.sum())
            valid = (negative_mass <= NEGATIVE_MASS_THRESHOLD
                     and excluded_fraction <= EXCLUDED_EMPIRICAL_MASS_THRESHOLD)
            row = summarize_approximation(
                result, conditioned_correction,
                counts_by_replica=conditioned_counts, n_total=retained_samples,
            )
            if not valid:
                row.update(KL_uncertainty=np.nan, replica_raw_KL_mean=np.nan,
                           replica_raw_KL_sem=np.nan)
            row.update(tau=tau, D=D, Approximation='Correction', outside=outside,
                       tau2D=D*tau**2, negative_mass=negative_mass,
                       nonpositive_theory_bins=int(np.sum(corrected <= 0)),
                       excluded_empirical_fraction=excluded_fraction,
                       correction_valid=valid)
            rows.append(row)
    results = pd.DataFrame(rows)
    results.sort_values(['Approximation', 'tau'])


## Stationary-density comparison


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    DISPLAY_TAUS = [0.2, 1.0, 5.0]
    VISUAL_I_MIN = 0.0
    VISUAL_I_MAX = 2.5
    print('Using packaged logistic density data.')
else:
    # Desired TOTAL samples in the visual comparison, production included.
    DISPLAY_TAUS = [0.2, 1.0, 5.0]
    VISUAL_OUTPUT_DIR = OUTPUT_DIR / 'stationary_visual_comparison'
    VISUAL_TARGET_TOTAL_SAMPLES = 10000000
    VISUAL_I_MIN = 0.0
    VISUAL_I_MAX = 2.5
    VISUAL_N_BINS = 35
    RUN_VISUAL_SIMULATIONS = False

    VISUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    visual_add_on_data = {}
    for tau in DISPLAY_TAUS:
        key = (tau, D_PRODUCTION)
        production = np.asarray(stationary_data[key]['replica_endpoints_I']).reshape(-1)
        required = max(0, VISUAL_TARGET_TOTAL_SAMPLES - production.size)
        filename = VISUAL_OUTPUT_DIR / f'add_on_tau_{tau:g}_D_{D_PRODUCTION:g}.pkl'
        if filename.exists():
            with filename.open('rb') as handle:
                saved = pickle.load(handle)
            add_on = np.asarray(saved['endpoints_I']).reshape(-1)
            seeds = list(saved.get('seeds', []))
        else:
            add_on = np.empty(0, dtype=float)
            seeds = []
        if RUN_VISUAL_SIMULATIONS and add_on.size < required:
            n_new = required - add_on.size
            config = LogisticExperiment(tau=tau)
            seed = int(np.random.SeedSequence([
                config.base_seed, int(round(tau * 1_000_000)),
                int(round(config.D * 1_000_000)), 0xF2A, add_on.size,
            ]).generate_state(1, dtype=np.uint32)[0])
            print(f'tau={tau:g}: simulating {n_new:,} additional endpoints')
            new_endpoints = simulate_logistic_endpoints(
                config.D, config.tau, config.dt, config.n_steps,
                n_new, seed, config.initial_I,
            )
            add_on = np.concatenate([add_on, new_endpoints])
            seeds.append(seed)
            temporary = filename.with_suffix('.pkl.tmp')
            with temporary.open('wb') as handle:
                pickle.dump({'endpoints_I': add_on, 'seeds': seeds}, handle)
            temporary.replace(filename)
        visual_add_on_data[key] = add_on
        print(
            f'tau={tau:g}: target={VISUAL_TARGET_TOTAL_SAMPLES:,}, '
            f'production={production.size:,}, required add-on={required:,}, '
            f'saved add-on={add_on.size:,}'
        )


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    density_plot_data = pd.read_csv(
        PLOT_DATA_DIR / 'logistic_density.csv'
    )

    fig = plt.figure(figsize=(7, 3.2))
    grid = fig.add_gridspec(len(DISPLAY_TAUS), 2, width_ratios=[2.5, 2])
    ax_pdf = fig.add_subplot(grid[:, 0])
    deviation_axes = [fig.add_subplot(grid[index, 1])
                      for index in range(len(DISPLAY_TAUS))]

    for tau, color, ax_dev in zip(
        DISPLAY_TAUS, sns.color_palette()[:len(DISPLAY_TAUS)], deviation_axes
    ):
        data = density_plot_data[
            np.isclose(density_plot_data['tau'], tau)
        ].sort_values('I')
        centers = data['I'].to_numpy()
        empirical = data['simulation'].to_numpy()
        empirical_sem = data['simulation_uncertainty'].to_numpy()
        ucna = data['UCNA'].to_numpy()
        corrected = data['Correction'].to_numpy()
        cbfpe = data['cBFPE'].to_numpy()
        label = fr'$\tau = {tau:g}$'

        ax_pdf.errorbar(
            centers, empirical, yerr=empirical_sem, fmt='none', ecolor=color,
            elinewidth=1.0, capsize=2.5, capthick=1.0, zorder=8,
        )
        ax_pdf.plot(
            centers, empirical, linestyle='none', marker='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            label=label, zorder=9,
        )
        ax_pdf.plot(centers, ucna, color=color, lw=1.4, label='UCNA, LLA')
        ax_pdf.plot(centers, corrected, color=color, ls='--', lw=1.5,
                    label='Correction')
        ax_pdf.plot(centers, cbfpe, color=color, ls='-.', lw=1.3,
                    label='cBFPE')

        ax_dev.axhline(0, color=color, alpha=.35, lw=1)
        deviation = empirical - ucna
        ax_dev.errorbar(
            centers, deviation, yerr=empirical_sem, fmt='none', ecolor=color,
            elinewidth=1.0, capsize=2.5, capthick=1.0, zorder=8,
        )
        ax_dev.plot(
            centers, deviation, linestyle='none', marker='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            zorder=9,
        )
        ax_dev.plot(centers, corrected-ucna, color=color, ls='--', lw=1.3)
        ax_dev.plot(centers, cbfpe-ucna, color=color, ls='-.', lw=1.2)
        scatter_min = np.min(deviation - empirical_sem)
        scatter_max = np.max(deviation + empirical_sem)
        scatter_span = scatter_max - scatter_min
        padding = 0.05 * scatter_span if scatter_span > 0 else 1e-3
        ax_dev.set_ylim(scatter_min-padding, scatter_max+padding)

    ax_pdf.set(xlabel='$I$', ylabel='Probability Density',
               xlim=(VISUAL_I_MIN, VISUAL_I_MAX))
    deviation_axes[1].set_ylabel('Deviation from UCNA')
    deviation_axes[-1].set_xlabel('$I$')
    for axis in deviation_axes:
        axis.set_xlim(VISUAL_I_MIN, VISUAL_I_MAX)
    for axis in deviation_axes[:-1]:
        axis.tick_params(labelbottom=False)
    display_colors = sns.color_palette()[:len(DISPLAY_TAUS)]
    legend_handles = [
        mpl.lines.Line2D([], [], color='black', ls='-', label='UCNA, LLA'),
        mpl.lines.Line2D([], [], color='black', ls='--', label='Correction'),
        mpl.lines.Line2D([], [], color='black', ls='-.', label='cBFPE'),
    ]
    for tau, color in zip(DISPLAY_TAUS, display_colors):
        legend_handles.append(ax_pdf.errorbar(
            [np.nan], [np.nan], yerr=[0.1], fmt='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            ecolor=color, elinewidth=1.0, capsize=2.5, capthick=1.0,
            label=fr'$\tau = {tau:g}$',
        ))
    ax_pdf.legend(handles=legend_handles, loc='upper right')
    fig.tight_layout()
    plt.show()
else:
    # Alternative stationary comparison including the logistic cBFPE.
    fig = plt.figure(figsize=(7, 3.2))
    grid = fig.add_gridspec(len(DISPLAY_TAUS), 2, width_ratios=[2.5, 2])
    ax_pdf = fig.add_subplot(grid[:, 0])
    deviation_axes = [fig.add_subplot(grid[index, 1])
                      for index in range(len(DISPLAY_TAUS))]

    for tau, color, ax_dev in zip(
        DISPLAY_TAUS, sns.color_palette()[:len(DISPLAY_TAUS)], deviation_axes
    ):
        result = stationary_data[(tau, D_PRODUCTION)]
        edges = np.linspace(VISUAL_I_MIN, VISUAL_I_MAX, VISUAL_N_BINS + 1)
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)
        production = np.asarray(result['replica_endpoints_I']).reshape(-1)
        n_production = min(production.size, VISUAL_TARGET_TOTAL_SAMPLES)
        required_add_on = VISUAL_TARGET_TOTAL_SAMPLES - n_production
        add_on = visual_add_on_data[(tau, D_PRODUCTION)]
        if add_on.size < required_add_on:
            raise RuntimeError(
                f'tau={tau:g} needs {required_add_on-add_on.size:,} more visual samples'
            )
        visual_endpoints = np.concatenate([
            production[:n_production], add_on[:required_add_on],
        ])
        pooled = np.histogram(visual_endpoints, bins=edges)[0]
        empirical = pooled / (pooled.sum() * widths)
        empirical_sem = np.sqrt(np.maximum(pooled, 1)) / (pooled.sum() * widths)

        theory_edges = edges.copy()
        if theory_edges[0] == 0.0:
            theory_edges[0] = logistic_intensity_interval(tau, D_PRODUCTION)[0]
        ucna = ucna_bin_probabilities(tau, D_PRODUCTION, theory_edges) / widths
        corrected_weights, _ = corrected_bin_probabilities(
            tau, D_PRODUCTION, theory_edges
        )
        corrected = corrected_weights / widths
        cbfpe = bfpe_bin_probabilities(tau, D_PRODUCTION, theory_edges) / widths

        label = fr'$\tau = {tau:g}$'
        ax_pdf.errorbar(
            centers, empirical, yerr=empirical_sem, fmt='none', ecolor=color,
            elinewidth=1.0, capsize=2.5, capthick=1.0, zorder=8,
        )
        ax_pdf.plot(
            centers, empirical, linestyle='none', marker='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            label=label, zorder=9,
        )
        ax_pdf.plot(centers, ucna, color=color, lw=1.4, label='UCNA, LLA')
        ax_pdf.plot(centers, corrected, color=color, ls='--', lw=1.5,
                    label='Correction')
        ax_pdf.plot(centers, cbfpe, color=color, ls='-.', lw=1.3,
                    label='cBFPE')

        ax_dev.axhline(0, color=color, alpha=.35, lw=1)
        deviation = empirical - ucna
        ax_dev.errorbar(
            centers, deviation, yerr=empirical_sem, fmt='none', ecolor=color,
            elinewidth=1.0, capsize=2.5, capthick=1.0, zorder=8,
        )
        ax_dev.plot(
            centers, deviation, linestyle='none', marker='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            zorder=9,
        )
        ax_dev.plot(centers, corrected-ucna, color=color, ls='--', lw=1.3)
        ax_dev.plot(centers, cbfpe-ucna, color=color, ls='-.', lw=1.2)
        # Scale this panel from the empirical scatter and its error bars only.
        scatter_min = np.min(deviation - empirical_sem)
        scatter_max = np.max(deviation + empirical_sem)
        scatter_span = scatter_max - scatter_min
        padding = 0.05 * scatter_span if scatter_span > 0 else 1e-3
        ax_dev.set_ylim(scatter_min-padding, scatter_max+padding)

    ax_pdf.set(xlabel='$I$', ylabel='Probability Density',
               xlim=(VISUAL_I_MIN, VISUAL_I_MAX))
    deviation_axes[1].set_ylabel('Deviation from UCNA')
    deviation_axes[-1].set_xlabel('$I$')
    for axis in deviation_axes:
        axis.set_xlim(VISUAL_I_MIN, VISUAL_I_MAX); #axis.grid(alpha=.15)
    for axis in deviation_axes[:-1]:
        axis.tick_params(labelbottom=False)
    display_colors = sns.color_palette()[:len(DISPLAY_TAUS)]
    legend_handles = [
        mpl.lines.Line2D([], [], color='black', ls='-', label='UCNA, LLA'),
        mpl.lines.Line2D([], [], color='black', ls='--', label='Correction'),
        mpl.lines.Line2D([], [], color='black', ls='-.', label='cBFPE'),
    ]
    for tau, color in zip(DISPLAY_TAUS, display_colors):
        legend_handles.append(ax_pdf.errorbar(
            [np.nan], [np.nan], yerr=[0.1], fmt='s', markersize=2.2,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=0.8,
            ecolor=color, elinewidth=1.0, capsize=2.5, capthick=1.0,
            label=fr'$\tau = {tau:g}$',
        ))
    ax_pdf.legend(handles=legend_handles, loc='upper right')
    fig.tight_layout()
    plt.show()


## KL-divergence figure


In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 3.2))
plot_styles = {
    'UCNA': dict(marker='o', linestyle='-',  ms=5, lw=1.5, zorder=3),
    'Correction': dict(marker='D', linestyle=':',  ms=5, lw=1.7, zorder=4),
    'cBFPE': dict(marker='s', linestyle='-', mfc='white', ms=6, lw=1.5, zorder=5),
}

for approximation in ['UCNA', 'Correction', 'cBFPE']:
    data = results.query('Approximation == @approximation').sort_values('tau')
    ax.plot(data.tau, data.KL, color='C0', mec='C0', mew=0.8,
            label=approximation, **plot_styles[approximation])
    lower = np.maximum(data.KL-data.KL_uncertainty, np.finfo(float).tiny)
    ax.fill_between(data.tau, lower, data.KL+data.KL_uncertainty,
                    color='C0', alpha=.25, linewidth=0)
invalid = results.query("Approximation == 'Correction' and not correction_valid")
ax.scatter(invalid.tau, invalid.KL, marker='X', s=65, color='C0',
           label='Non-positive correction', zorder=5)
ax.axhline(.5, color='grey', alpha=.5)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xticks(TAUS, [f'{tau:g}' for tau in TAUS], minor=False)
ax.set(xlabel='$\mathbf{\\tau}$', ylabel='$\mathbf{d_{KL}}$')
handles, labels = ax.get_legend_handles_labels()
order = [labels.index(name) for name in ['UCNA', 'Correction', 'cBFPE']]
if len(invalid) > 0:
    order.append(labels.index('Non-positive correction'))
ax.legend([handles[index] for index in order], [labels[index] for index in order], loc = (0.45,0.5))
ax.annotate(r'$\left(b\right)$', (0.18, 5000), fontsize=15, fontweight='bold')
#ax.set_ylim(0.1,100)
fig.tight_layout()
plt.show()
